In [1]:
import pandas as pd
import numpy as np
import pickle

prices = pd.read_csv("../data/processed/prices.csv", index_col="date", parse_dates=True)
with open("../data/processed/pair_signals.pkl", "rb") as f:
    all_signals = pickle.load(f)

selected_pairs = pd.read_csv("../data/processed/selected_pairs.csv")
print(f"Loaded signals for {len(all_signals)} pairs")

Loaded signals for 87 pairs


In [2]:
def generate_positions(z_score, entry=1.0, exit=0.2):
    """
    Returns a position series: +1 = long coin_a / short coin_b,
    -1 = short coin_a / long coin_b, 0 = flat.
    """
    position = pd.Series(0, index=z_score.index)
    pos = 0
    for i in range(len(z_score)):
        z = z_score.iloc[i]
        if pd.isna(z):
            position.iloc[i] = 0
            continue
        if pos == 0:
            if z > entry:
                pos = -1   # spread too high -> bet it falls back: short a, long b
            elif z < -entry:
                pos = 1    # spread too low -> bet it rises back: long a, short b
        else:
            if abs(z) < exit:
                pos = 0    # spread back near normal -> close out
        position.iloc[i] = pos
    return position

In [3]:
EXIT_THRESHOLDS = [0.1, 0.2, 0.5, 0.7]

def summarize_positions(position):
    """Quick stats: number of trades entered, average holding length."""
    changes = position.diff().fillna(0) != 0
    n_trades = (changes & (position != 0)).sum()
    days_in_position = (position != 0).sum()
    avg_holding = days_in_position / n_trades if n_trades > 0 else 0
    return n_trades, avg_holding

# test on the single best pair by cointegration p-value first
best_pair = tuple(selected_pairs.sort_values("p_value").iloc[0][["coin_a", "coin_b"]])
z = all_signals[best_pair]["z_score"]

print(f"Pair: {best_pair}")
for exit_thresh in EXIT_THRESHOLDS:
    pos = generate_positions(z, entry=1.0, exit=exit_thresh)
    n_trades, avg_holding = summarize_positions(pos)
    print(f"  exit={exit_thresh}: {n_trades} trades, {avg_holding:.1f} days avg holding")

Pair: ('havven', 'jasmycoin')
  exit=0.1: 12 trades, 31.7 days avg holding
  exit=0.2: 16 trades, 21.6 days avg holding
  exit=0.5: 22 trades, 11.3 days avg holding
  exit=0.7: 22 trades, 10.3 days avg holding


In [4]:
EXIT_THRESHOLD = 0.2  # a reasonable middle ground; revisit after Phase 8's comparison

all_positions = {}
for pair, sig in all_signals.items():
    all_positions[pair] = generate_positions(sig["z_score"], entry=1.0, exit=EXIT_THRESHOLD)

print(f"Generated positions for {len(all_positions)} pairs")

Generated positions for 87 pairs


In [9]:
def compute_pair_returns(price_a, price_b, position, beta):
    """Daily P&L for a dollar-neutral pair position, sized by hedge ratio."""
    ret_a = price_a.pct_change(fill_method=None)
    ret_b = price_b.pct_change(fill_method=None)
    # long/short a, hedge-ratio-weighted short/long b
    pair_return = position.shift(1) * (ret_a - beta.shift(1) * ret_b)
    return pair_return

pair_returns = {}
for pair, sig in all_signals.items():
    a, b = pair
    pair_returns[pair] = compute_pair_returns(prices[a], prices[b], all_positions[pair], sig["beta"])

# equal-weight across all 87 pairs for now
portfolio_returns = pd.DataFrame(pair_returns).mean(axis=1)

In [10]:
portfolio_returns.describe()

count    641.000000
mean       0.000390
std        0.006692
min       -0.081378
25%       -0.001801
50%        0.000000
75%        0.003458
max        0.048569
dtype: float64

In [11]:
portfolio_returns.dropna().head()

date
2024-11-25    0.0
2024-11-26    0.0
2024-11-27    0.0
2024-11-28    0.0
2024-11-29    0.0
dtype: float64

In [12]:
portfolio_returns.dropna()[portfolio_returns.dropna() != 0].head()

date
2025-02-22    0.005284
2025-02-23   -0.006926
2025-02-24   -0.020068
2025-02-25    0.006138
2025-02-26   -0.005085
dtype: float64

In [13]:
portfolio_returns.to_csv("../data/processed/portfolio_returns.csv")